## **Setup**

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]  # notebooks/ is one level down
sys.path.append(str(PROJECT_ROOT / "src"))

In [4]:
from chatGnT.config import CFG, ensure_dirs
import chatGnT.utils as utils
import chatGnT.data.load as load
import chatGnT.data.preprocess as preprocess
import chatGnT.data.tokenize as tokenize
ensure_dirs(CFG)
import time
import math
import torch
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn  # for embedding layer
from torch.utils.data import TensorDataset, DataLoader
import chatGnT.models.transformer as transformer
import chatGnT.models.positional_encoding as positional_encoding
import chatGnT.models.train as train
import chatGnT.models.evaluate as evaluate
import chatGnT.models.predict as predict

/Users/slacksa/miniconda3/envs/chatGnT/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **Load & Clean Data**

In [5]:
data = load.load_all()
df = data["ingred"]
#TODO: revisit and handle things like "Twist of  Orange peel" "Top with..."
#TODO: how make units consistent - good opportunity for test functions?
df_clean = preprocess.clean_ingred(df)
print(df_clean.head())
print(df.shape)
print(df_clean.shape)


   id     ingredient_name               ingredient_link  \
0   1   1 oz  Coconut rum   /ingredient/135-Coconut-rum   
1   1    1/2 oz  Amaretto       /ingredient/18-Amaretto   
2   1  4 oz  Orange juice  /ingredient/352-Orange-juice   
3   1   1/2 oz  Grenadine     /ingredient/250-Grenadine   
4   2     2 oz  Light rum     /ingredient/305-Light-rum   

                                ingredient_image  amt unit        ingred  
0   ../images/ingredients/Coconut rum-Medium.png    1   oz   Coconut rum  
1      ../images/ingredients/Amaretto-Medium.png  1/2   oz      Amaretto  
2  ../images/ingredients/Orange juice-Medium.png    4   oz  Orange juice  
3     ../images/ingredients/Grenadine-Medium.png  1/2   oz     Grenadine  
4     ../images/ingredients/Light rum-Medium.png    2   oz     Light rum  
(2491, 4)
(1075, 7)


## **Make Vocab & Tokenize**

In [6]:
tokens = tokenize.recipe_to_tokens(df_clean)
print(tokens[0])

['<amt>1</amt>', '<unit>oz</unit>', '<ingred>Coconut rum</ingred>', '<sep>', '<amt>1/2</amt>', '<unit>oz</unit>', '<ingred>Amaretto</ingred>', '<sep>', '<amt>4</amt>', '<unit>oz</unit>', '<ingred>Orange juice</ingred>', '<sep>', '<amt>1/2</amt>', '<unit>oz</unit>', '<ingred>Grenadine</ingred>', '<sep>']


In [7]:
vocab = tokenize.make_vocab(tokens)
print(len(vocab))  # 611 classes to predict
inv_vocab = tokenize.invert_vocab(vocab)
tokens_padded = tokenize.embed_tokens(tokens, vocab)

437


In [8]:
# tokens_tensor = [torch.tensor(r) for r in tokens_padded]
tokens_tensor = torch.tensor(tokens_padded, dtype=torch.long)
print(tokens_tensor.shape)  # torch.Size([621, 48])

# Need to have [seq_len, batch_size] for TransformerEncoder
# tokens_tensor = tokens_tensor.transpose(0, 1)
# print(tokens_tensor.shape)  # torch.Size([49, 621])

torch.Size([306, 33])


In [9]:
print(len(tokens_padded))
type(tokens_padded)
print(tokens_padded[0])

306
[19, 413, 111, 369, 9, 413, 53, 369, 32, 413, 223, 369, 9, 413, 168, 369, 436, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [10]:
#TODO: need to do this elsewhere so matches batch size?


# TODO: for now, add padding mask but not a causal mask that block seeing future
pad_id = vocab["<pad>"]   # should be 0
# src_key_padding_mask shape = [batch_size, seq_len]
pad_mask = (tokens_tensor == pad_id).transpose(0, 1)
print(pad_mask.shape)   # [batch_size, seq_len]


torch.Size([33, 306])


## **Get Batches**

In [11]:
# Not shifting targets since want model to attend to all tokens
inputs = tokens_tensor
targets = tokens_tensor

dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

print(dataloader.batch_size)  # 32
print(len(dataloader))  # 20 = 621 / 32

32
10


## **Train Model**

In [12]:
embed_size=16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = transformer.TransformerModel(
    ntoken=len(vocab),
    ninp=embed_size,
    nhead=4,
    nhid=256,
    nlayers=2).to(device)
#TODO: investigate error? 

learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss(ignore_index=pad_id)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.95)
#TODO: read more about schedular

/Users/slacksa/repos/chatGnT/src/chatGnT/models/transformer.py:17: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers)


In [13]:
best_val_loss = float("inf")
epochs = 12  # number of epochs
best_model = None

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()

    train.train(model, dataloader, device, pad_id, optimizer, criterion, epoch, 6)
    val_loss = evaluate.evaluate(model, dataloader, device, pad_id, criterion)

    print('-' * 89)
    print(
        f'Epoch {epoch} | Val Loss: {val_loss:.4f} | '
        f'Time {(time.time() - epoch_start_time)} | Val PPL: {math.exp(val_loss):.2f}')
        #TODO: currently not retruning training loss - should I? When look at each?
    print('-' * 89)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = model

    scheduler.step()  # adjusts learning rate
    #TODO: read more about this?


/Users/slacksa/miniconda3/envs/chatGnT/lib/python3.11/site-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


Epoch 1 | Batch 0 | LR 0.001000 | Loss 1.0187 | PPL 2.77 | Time 0.06s
Epoch 1 | Batch 6 | LR 0.001000 | Loss 6.0108 | PPL 407.82 | Time 0.13s
-----------------------------------------------------------------------------------------
Epoch 1 | Val Loss: 5.7818 | Time 0.2982149124145508 | Val PPL: 324.36
-----------------------------------------------------------------------------------------
Epoch 2 | Batch 0 | LR 0.000950 | Loss 0.9741 | PPL 2.65 | Time 0.02s
Epoch 2 | Batch 6 | LR 0.000950 | Loss 5.7994 | PPL 330.11 | Time 0.13s
-----------------------------------------------------------------------------------------
Epoch 2 | Val Loss: 5.6150 | Time 0.25539493560791016 | Val PPL: 274.52
-----------------------------------------------------------------------------------------
Epoch 3 | Batch 0 | LR 0.000902 | Loss 0.9468 | PPL 2.58 | Time 0.02s
Epoch 3 | Batch 6 | LR 0.000902 | Loss 5.6594 | PPL 286.98 | Time 0.13s
-----------------------------------------------------------------------

## **Evaluate with Test Data**

In [14]:
#TODO: for now, use same data for test but should have separate test set
test_data = dataloader

test_loss = evaluate.evaluate(best_model, test_data, device, pad_id, criterion)
print('=' * 89)
print(f'Test Loss: {test_loss:5.2f} | Test PPL: {math.exp(test_loss):8.2f}')
print('=' * 89)

Test Loss:  4.23 | Test PPL:    69.06


## **Run with Test User Input**

In [15]:
#TODO: do need to do general string cleaning... Absinthe vs. absinthe
# <ingred>Absinthe</ingred>': 59
input = "Absinthe"

# tokenize
#TODO: think a different function would be better? Or just need function to handle all
# formats of user input to this format?
# input: df (pd.DataFrame): Must have columns ['amount', 'unit', 'ingred']
#TODO: for now just manually making token
# input_mod = "<ingred>Absinthe</ingred>"
input_mod = ['<amt>1</amt>', '<unit>oz</unit>', '<ingred>Gin</ingred>', '<sep>']

In [18]:
#TODO: how to embed?
#TODO: consider adding unknown ingredient token for things not in vocab?
predict.predict(best_model, device, pad_id, vocab, inv_vocab, input_mod)




['<amt>1</amt>',
 '<unit>oz</unit>',
 '<ingred>Gin</ingred>',
 '<sep>',
 '<ingred>Sarsaparilla</ingred>',
 '<unit>oz</unit>',
 '<unit>shots</unit>',
 '<unit>oz</unit>',
 '<ingred>Tabasco sauce</ingred>',
 '<ingred>Lime</ingred>',
 '<unit>Guinness</unit>',
 '<end>',
 '<ingred>Grapefruit Juice</ingred>',
 '<unit>Egg</unit>',
 '<unit>shot</unit>',
 '<ingred>Water</ingred>',
 '<ingred>Gin</ingred>',
 '<unit>small</unit>',
 '<unit>oz</unit>',
 '<sep>',
 '<ingred>Triple sec</ingred>',
 '<unit>oz</unit>',
 '<end>']

In [17]:
predict.predict_groups(best_model, device, pad_id, vocab, inv_vocab, input_mod)

['<amt>1</amt>',
 '<unit>oz</unit>',
 '<ingred>Gin</ingred>',
 '<sep>',
 '<ingred>Sarsaparilla</ingred>',
 '<unit>oz</unit>',
 '<unit>shots</unit>',
 '<unit>oz</unit>',
 '<ingred>Tabasco sauce</ingred>',
 '<ingred>Lime</ingred>',
 '<unit>Guinness</unit>',
 '<end>']